#### **Install Dependencies**

In [1]:
! pip install -q llama-index-core llama-index-embeddings-huggingface llama-index-postprocessor-sbert-rerank llama-index-llms-openai sentence-transformers datasets

#### **Load Dataset**

In [2]:
from datasets import load_dataset

# load Amharic Question Answering dataset
amharic_qa = load_dataset("israel/AmharicQA", split="validation").shuffle(seed=42)
amharic_qa

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 595
})

In [3]:
queries = amharic_qa['question']
documents = amharic_qa['context']
answers = amharic_qa['answers']

len(queries), len(documents), len(answers)

(595, 595, 595)

#### **Two Stage Retrieval with Llama Index**

In [4]:
from llama_index.core import Document, VectorStoreIndex, Settings, QueryBundle
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.postprocessor.sbert_rerank import SentenceTransformerRerank
from llama_index.core.schema import TextNode

In [5]:
# explicitly disable the LLM
Settings.llm = None

nodes = [
    TextNode(text=document, metadata={})
    for document in list(set(documents))
]

print('Number of Nodes:', len(nodes))

LLM is explicitly disabled. Using MockLLM.
Number of Nodes: 56


In [ ]:
# 1. Configure Embedding
Settings.embed_model = HuggingFaceEmbedding(
    model_name="rasyosef/embedding-amharic-base"
)

# 2. Build Index
index = VectorStoreIndex(nodes, show_progress=True)

In [ ]:
# 3. Setup Retriever and Reranker independently
retriever = index.as_retriever(similarity_top_k=5)
reranker = SentenceTransformerRerank(
    model="rasyosef/reranker-amharic-base",
    top_n=3,
    cross_encoder_kwargs={"max_length": 510} # Forces the tokenizer to clip the input
  )

In [17]:
idx = 6
query = queries[idx]
print(query)

በአጼ ቀዳማዊ ኃይለ ሥላሴ ታውጆ እስከ 1948 አ.ም. ሲያገለግል የነበረው ሕገ መንግሥት መች ነበር የታወጀው?


In [18]:
# 4. Retrieve initial candidates (Vector Search)
retrieved_nodes = retriever.retrieve(query)

print(f"Query: {query}\n")
print("--- Retrieved Source Nodes ---")
for node in retrieved_nodes:
    print(f"Score: {node.score:.3f} | Text: {node.text}...")

Query: በአጼ ቀዳማዊ ኃይለ ሥላሴ ታውጆ እስከ 1948 አ.ም. ሲያገለግል የነበረው ሕገ መንግሥት መች ነበር የታወጀው?

--- Retrieved Source Nodes ---
Score: 0.881 | Text: በ1240 አካባቢ በግብጽ አገር ቅብጡ አቡል ፋዳዒል እብን አል-አሣል ፍትሐ ነገሥት በአረብኛ ጻፉ። እብን አል-አሣል ሕጎቹን የወሰዱት በከፊል ከሃዋርያት ጽህፈቶችና ከሕገ ሙሴ፤ በከፊልም ከድሮ ቢዛንታይን ነገስታት ሕጎች ነበር። መጽሐፉ በግዕዝ ተተርጉሞ ወደ ኢትዮጵያ የገባበት ወቅት በዐፄ ዘርዐ ያዕቆብ ዘመን በ1450 አከባቢ እንደነበር የሚል ታሪካዊ መዝገብ አለ።  ቢሆንም መጀመርያ እንደ ሕገ ምንግሥት መጠቀሙን የተመዘገበው በዓፄ ሠርፀ ድንግል ዘመን (ከ1555 ጀምሮ) ነው። ፍትሐ ነገስት እስከ 1923 ዓ.ም. የብሄሩን ዋና ሕግ ሆኖ ቆየ። በ1904 ዓ.ም. ዐጼ 2 ምኒልክ የዘመናዊ ሕገ መንግሥት ፅንሰ ሀሳብ ተረድተው በጸሀፊው መምሬ ብስራት አንድ ሕገ መንግሥት የሚመስል ሰነድ ታተመ፤ ይህ ግን በውነት ሕገ መንግሥት አይባልም። «በምኒሊክ ስለተቋቋሙት ሚኒስቴሮች በምሳሌ የቀረበ ጽሑፍ» በምሳሌና በትርጓሜ የእያንዳንዱን ሚኒስቴር ሥራ እንደ ሰውነት አካላት አስመሰለ። ለምሳሌ፦ ኢትዮጵያ መጀመርያ ዘመናዊ ሕገ መንግሥት የተቀበለችው በ1923 አመተ ምኅረት በአጼ ቀዳማዊ ኃይለ ሥላሴ የታወጀ ሲሆን የሕግ አማካሪ ቤቶች በሁለት ያስከፈለ ነው።  ይህ የመንግሥታቸው ዋና መሠረት ሆኖ እስከ 1948 አ.ም. አገለገለ፤ በዚያ አመት ተሻሽሎ በወጣ ሕገ መንግሥት ሕዝቦች በመንግሥት ሥራ የሚጫወቱት ሚና እንደገና ተስፋፋ።  ይህ ብቻ የአገሪቱ ሕገ መንግሥት ሆኖ እስከ 1967 አ.ም. ቆየ፤ የዛኔ በደርግ (በሕገ መንግስት እራሱ ባልሆነ ሂደት) ተሰረዘ።...
Score: 0.528

In [19]:
# 5. Rerank the candidates (Cross-Encoder)
reranked_nodes = reranker.postprocess_nodes(
    retrieved_nodes,
    query_bundle=QueryBundle(query)
)

print(f"Query: {query}\n")
print("--- Reranked Source Nodes ---")
for node in reranked_nodes:
    print(f"Score: {node.score:.3f} | Text: {node.text}")

Query: በአጼ ቀዳማዊ ኃይለ ሥላሴ ታውጆ እስከ 1948 አ.ም. ሲያገለግል የነበረው ሕገ መንግሥት መች ነበር የታወጀው?

--- Reranked Source Nodes ---
Score: 0.996 | Text: በ1240 አካባቢ በግብጽ አገር ቅብጡ አቡል ፋዳዒል እብን አል-አሣል ፍትሐ ነገሥት በአረብኛ ጻፉ። እብን አል-አሣል ሕጎቹን የወሰዱት በከፊል ከሃዋርያት ጽህፈቶችና ከሕገ ሙሴ፤ በከፊልም ከድሮ ቢዛንታይን ነገስታት ሕጎች ነበር። መጽሐፉ በግዕዝ ተተርጉሞ ወደ ኢትዮጵያ የገባበት ወቅት በዐፄ ዘርዐ ያዕቆብ ዘመን በ1450 አከባቢ እንደነበር የሚል ታሪካዊ መዝገብ አለ።  ቢሆንም መጀመርያ እንደ ሕገ ምንግሥት መጠቀሙን የተመዘገበው በዓፄ ሠርፀ ድንግል ዘመን (ከ1555 ጀምሮ) ነው። ፍትሐ ነገስት እስከ 1923 ዓ.ም. የብሄሩን ዋና ሕግ ሆኖ ቆየ። በ1904 ዓ.ም. ዐጼ 2 ምኒልክ የዘመናዊ ሕገ መንግሥት ፅንሰ ሀሳብ ተረድተው በጸሀፊው መምሬ ብስራት አንድ ሕገ መንግሥት የሚመስል ሰነድ ታተመ፤ ይህ ግን በውነት ሕገ መንግሥት አይባልም። «በምኒሊክ ስለተቋቋሙት ሚኒስቴሮች በምሳሌ የቀረበ ጽሑፍ» በምሳሌና በትርጓሜ የእያንዳንዱን ሚኒስቴር ሥራ እንደ ሰውነት አካላት አስመሰለ። ለምሳሌ፦ ኢትዮጵያ መጀመርያ ዘመናዊ ሕገ መንግሥት የተቀበለችው በ1923 አመተ ምኅረት በአጼ ቀዳማዊ ኃይለ ሥላሴ የታወጀ ሲሆን የሕግ አማካሪ ቤቶች በሁለት ያስከፈለ ነው።  ይህ የመንግሥታቸው ዋና መሠረት ሆኖ እስከ 1948 አ.ም. አገለገለ፤ በዚያ አመት ተሻሽሎ በወጣ ሕገ መንግሥት ሕዝቦች በመንግሥት ሥራ የሚጫወቱት ሚና እንደገና ተስፋፋ።  ይህ ብቻ የአገሪቱ ሕገ መንግሥት ሆኖ እስከ 1967 አ.ም. ቆየ፤ የዛኔ በደርግ (በሕገ መንግስት እራሱ ባልሆነ ሂደት) ተሰረዘ።
Score: 0.003 | T

### **RAG**

In [20]:
import os
from google.colab import userdata

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [21]:
from llama_index.llms.openai import OpenAI

# 1. Initialize your LLM
# Ensure your environment is configured for the OpenAI API
llm = OpenAI(model="gpt-5.4-mini")

# 2. Configure the Query Engine
# Passing these arguments constructs a pipeline that performs
# Retrieval -> Reranking -> Synthesis
query_engine = index.as_query_engine(
    node_postprocessors=[reranker],
    llm=llm
)

# 3. Query
response = query_engine.query(query)
print(f"Query: {query}\n")
print("--- --------------- ---")
print(f"Response: {response}\n")

Query: በአጼ ቀዳማዊ ኃይለ ሥላሴ ታውጆ እስከ 1948 አ.ም. ሲያገለግል የነበረው ሕገ መንግሥት መች ነበር የታወጀው?

--- --------------- ---
Response: 1923 አመተ ምኅረት ነበር።



In [22]:
print(f"Correct Answer: {answers[idx]}")

Correct Answer: 1923 ዓ.ም.
